# PyIceberg Workshop: Understanding Read & Write Paths


This hands-on notebook walks through the core operations of PyIceberg — Python's native implementation for Apache Iceberg (no JVM required).

---

## What This Notebook Covers

| # | Section | What You'll Do | Key Concept |
|---|---------|---------------|-------------|
| 1 | **Connect to Catalog** | Connect to REST Catalog (PostgreSQL + MinIO) | Catalog = the "phone book" for your tables |
| 2 | **Create Namespace** | Create a logical grouping for tables | Namespaces are metadata-only (PostgreSQL, not S3) |
| 3 | **Define Schema & Create Table** | Define 4-field schema, create empty table | Field IDs (not names) track column identity |
| 4 | **Write Path — Append Data** | Append 3 batches → 3 snapshots | Each append = Parquet + Manifest + Manifest List + Atomic Commit |
| 5 | **Inspect What Got Written** | Explore snapshots, manifests, data files, column stats | Understand the metadata tree that powers fast reads |
| 6 | **Read Path — Scan & Filter** | Full scan, filtered scan, column projection, time travel | The 3-layer pruning funnel: manifests → file stats → Parquet |
| 7 | **Schema Evolution** | Add a column, read old data seamlessly | Metadata-only change — zero data rewriting |
| 8 | **Snapshot Management** | Use the inspect API to query metadata as tables | `table.inspect.snapshots()`, `.manifests()`, `.files()` |

---

## How Each Example Is Structured

Every example follows the same pattern:

1. **What we're doing** — a brief description of the goal
2. **Code** — the cell you run
3. **Under the hood** — what PyIceberg does internally, including the **code path** through the source tree
4. **Expected output** — what you should see

---

## Prerequisites

- pip install -r requirements.txt
- Docker stack running: `docker-compose up -d` (REST Catalog + PostgreSQL + MinIO)
- Python packages: `pip install "pyiceberg[pyarrow]" pyarrow`
- MinIO UI available at http://localhost:9001 (admin/password) to browse S3 files

In [ ]:
#!pip install -r requirements.txt

---
## ⚙️ Prerequisites & Workshop Setup

### What's Running (Docker Stack)

Start the stack before running this notebook:

```bash
docker-compose up -d
```

| Service | URL | Purpose |
|---------|-----|---------|
| REST Catalog | http://localhost:8181 | Iceberg catalog — tracks table locations |
| MinIO S3 | http://localhost:9000 | Object storage — Parquet + metadata files |
| MinIO Console | http://localhost:9001 | Browse S3 files visually in browser |
| PostgreSQL | localhost:5432 | Catalog backend — stores snapshot state |

### What We're Building

A table called **`sensor_readings`** with this schema:

| Field | Type | Example |
|-------|------|---------|
| `sensor_id` | string | `PICARRO-001` |
| `ts` | timestamp | `2024-01-01 00:00:00` |
| `co2_ppm` | double | `405.3` |
| `ch4_ppb` | double | `1872.1` |

### Workshop Flow
1. Connect to REST Catalog
2. Create namespace + table
3. Append 3 batches (50 rows each) → 3 snapshots
4. Inspect: snapshots, manifests, data files
5. Read: filters, column projection, time travel
6. Evolve schema, inspect metadata

> **Run the cell below first** to confirm all services are reachable before we start.

In [ ]:
# ─── Infrastructure Health Check ────────────────────────────────────────────
# Run this first! Confirms all Docker services are reachable before we start.

import urllib.request
import urllib.error

checks = [
    ("REST Catalog",  "http://localhost:8181/v1/config"),
    ("MinIO S3",      "http://localhost:9000/minio/health/live"),
    ("MinIO Console", "http://localhost:9001"),
]

print("Checking workshop infrastructure...\n")
all_ok = True

for name, url in checks:
    try:
        urllib.request.urlopen(url, timeout=3)
        print(f"  ✅  {name:<20s}  UP    {url}")
    except urllib.error.HTTPError as e:
        # HTTPError means the server responded — service is up
        print(f"  ✅  {name:<20s}  UP    {url}")
    except Exception as e:
        print(f"  ❌  {name:<20s}  DOWN  — {e}")
        all_ok = False

print()
if all_ok:
    print("✅  All services are up — ready to start the workshop!")
else:
    print("❌  One or more services are down.")
    print("    Run:  docker-compose up -d")
    print("    Then wait ~10 seconds and re-run this cell.")

---
## 1. Connect to REST Catalog

**What we're doing:** Creating a client that can talk to our Iceberg REST Catalog. The catalog is the central registry that knows where every table's metadata lives. Think of it as a phone book — it maps table names to S3 locations.

In [ ]:
from pyiceberg.catalog.rest import RestCatalog

catalog = RestCatalog(
    name="workshop",
    **{
        "uri": "http://localhost:8181",
        "s3.endpoint": "http://localhost:9000",
        "s3.access-key-id": "admin",
        "s3.secret-access-key": "password",
        "s3.path-style-access": "true",
        "py-io-impl": "pyiceberg.io.pyarrow.PyArrowFileIO",
    },
)
print("Connected to REST Catalog")

**Expected output:**
```
Connected to REST Catalog
```

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py

    Code->>RC: RestCatalog(name, uri, **s3_config)
    RC->>RC: __init__() stores uri + s3 config in memory
    RC-->>Code: RestCatalog object (no HTTP yet)

    Note over Code,RC: Connection is LAZY — no network call until you use the catalog
```

**What just happened:**
- `RestCatalog.__init__()` simply stores the URI and S3 credentials in memory
- No HTTP request is made — PyIceberg uses **lazy connection** (connects on first use)
- `py-io-impl` tells PyIceberg to use `PyArrowFileIO` for all file I/O (reading Parquet, Avro)
- The first real network call happens when you call `list_namespaces()` or `create_table()`

---
## 2. Create Namespace

**What we're doing:** Creating a logical grouping called `pyiceberg_demo` for our tables. Namespaces are like schemas in a database — they organize tables but don't create anything on S3.

In [ ]:
NS = ("pyiceberg_demo",)

if NS not in catalog.list_namespaces():
    catalog.create_namespace(NS)
    print(f"Namespace '{NS[0]}' created")
else:
    print(f"Namespace '{NS[0]}' already exists")

**Expected output:**
```
Namespace 'pyiceberg_demo' created
```
*(or `already exists` if re-running)*

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py
    participant SRV as REST Catalog Server<br/><br/>localhost:8181
    participant PG as PostgreSQL<br/><br/>Metadata Store

    Code->>RC: catalog.list_namespaces()
    RC->>SRV: GET /v1/namespaces
    SRV->>PG: SELECT namespaces
    PG-->>SRV: []
    SRV-->>RC: {"namespaces": []}
    RC-->>Code: [] → not found, proceed

    Code->>RC: catalog.create_namespace("pyiceberg_demo")
    RC->>SRV: POST /v1/namespaces<br/>{"namespace": ["pyiceberg_demo"]}
    SRV->>PG: INSERT namespace row
    PG-->>SRV: ✅ done
    SRV-->>RC: 200 OK
    RC-->>Code: None

    Note over Code,PG: No S3 / MinIO involved — namespace is metadata only
```

**What just happened:**
- `catalog.list_namespaces()` makes the **first HTTP call** — `GET /v1/namespaces`
- `catalog.create_namespace()` sends `POST /v1/namespaces` with `{"namespace": ["pyiceberg_demo"]}`
- The catalog server inserts one row in PostgreSQL's `iceberg_namespace` table
- **Nothing is written to MinIO/S3** — namespaces are pure metadata
- Notice `schema.py` is NOT involved here — no field IDs, no schema, just a name

---
## 3. Define Schema & Create Table

**What we're doing:** Defining a 4-field schema for a `sensor_readings` table and creating it in the catalog. The fields represent Picarro gas analyzer data: sensor ID, timestamp, CO2 (ppm), and CH4 (ppb).

In [ ]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    NestedField, StringType, DoubleType, TimestampType,
)

TABLE_ID = (*NS, "sensor_readings")

schema = Schema(
    NestedField(1, "sensor_id", StringType()),
    NestedField(2, "ts",        TimestampType()),
    NestedField(3, "co2_ppm",   DoubleType()),
    NestedField(4, "ch4_ppb",   DoubleType()),
)

# Drop if exists (clean slate for workshop)
try:
    catalog.load_table(TABLE_ID)
    catalog.drop_table(TABLE_ID)
    print("(dropped existing table for clean run)")
except Exception:
    pass

table = catalog.create_table(TABLE_ID, schema=schema)
print(f"Table '{'.'.join(TABLE_ID)}' created")
print(f"Schema: {table.schema()}")

**Expected output:**
```
(dropped existing table for clean run)      ← only if re-running
Table 'pyiceberg_demo.sensor_readings' created
Schema: table {
  1: sensor_id: optional string
  2: ts: optional timestamp
  3: co2_ppm: optional double
  4: ch4_ppb: optional double
}
```

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant RC as RestCatalog<br/><br/>catalog/rest/__init__.py
    participant SC as schema.py<br/><br/>assign_fresh_schema_ids()
    participant SRV as REST Catalog Server<br/><br/>localhost:8181
    participant S3 as MinIO / S3<br/><br/>Object Storage
    participant PG as PostgreSQL<br/><br/>Metadata Store

    Code->>RC: catalog.create_table(identifier, schema, partition_spec)

    Note over RC,SC: ── Client-side only — no network calls yet ──

    RC->>SC: assign_fresh_schema_ids(schema)
    SC->>SC: Traverse each field<br/>assign monotonically increasing IDs
    SC-->>RC: fresh_schema<br/>sensor_id→1  ts→2  co2_ppm→3  ch4_ppb→4
    RC->>RC: build CreateTableRequest<br/>{name, fresh_schema, partition_spec, properties}

    Note over RC,PG: ── Network: POST to REST Catalog ──

    RC->>SRV: POST /v1/namespaces/pyiceberg_demo/tables
    SRV->>S3: Write metadata/00000-uuid.metadata.json<br/>{schema, field IDs, empty snapshots:[]}
    S3-->>SRV: ✅ written
    SRV->>PG: INSERT table record<br/>(namespace, name, metadata_location)
    PG-->>SRV: ✅ done
    SRV-->>RC: 200 OK + TableResponse<br/>{metadata_location, metadata, config}

    Note over RC,SC: ── Client-side: build Table object ──

    RC->>RC: _response_to_table()<br/>Table(identifier, metadata, FileIO → S3)
    RC-->>Code: Table object (empty — no data yet)

    Note over Code,PG: Field IDs 1-4 are now PERMANENT — renaming a column never changes its ID
```

**What just happened:**
- `RestCatalog.create_table()` first calls `assign_fresh_schema_ids(schema)` in `schema.py` — **100% client-side**, no network yet
- Each field gets a **permanent integer ID**: `sensor_id→1`, `ts→2`, `co2_ppm→3`, `ch4_ppb→4`
- Only then does PyIceberg send `POST /v1/namespaces/pyiceberg_demo/tables` to the REST server
- The server writes **exactly one file** to S3: `metadata/00000-<uuid>.metadata.json` — no `data/` folder yet
- The server registers the table in PostgreSQL, then returns the metadata
- **Field IDs are permanent** — if you later rename `co2_ppm` → `co2_concentration`, its ID stays `3`. That's how Iceberg tracks schema evolution safely across millions of files

### Verify empty table metadata

**What we're doing:** Inspecting the freshly created table to confirm it exists but has no data yet.

In [ ]:
import json

meta = table.metadata
print(f"Format version : {meta.format_version}")
print(f"Table UUID     : {meta.table_uuid}")
print(f"Location       : {meta.location}")
print(f"Current snap   : {meta.current_snapshot_id}")
print(f"Snapshots      : {len(meta.snapshots)}")
print(f"\nMetadata file on S3:\n  {table.metadata_location}\n")

with table.io.new_input(table.metadata_location).open() as f:
    print(json.dumps(json.loads(f.read()), indent=2))

**Reading the metadata.json — what each field means:**

```jsonc
{
  "format-version": 2,              ← Iceberg V2 (supports delete files, row-level deletes)
  "table-uuid": "6cf047cd-...",     ← permanent identity — never changes even if renamed
  "location": "s3://warehouse/...", ← root S3 path — all data + metadata lives under here
  "last-column-id": 4,              ← next new column gets ID 5
  "current-schema-id": 0,           ← which entry in "schemas" array is active
  "schemas": [{ "fields": [
      {"id": 1, "name": "sensor_id"},  ← field IDs are PERMANENT — rename won't break old reads
      {"id": 2, "name": "ts"},
      {"id": 3, "name": "co2_ppm"},
      {"id": 4, "name": "ch4_ppb"}
  ]}],
  "partition-specs": [{"fields": []}],  ← empty = unpartitioned
  "sort-orders":     [{"fields": []}],  ← empty = unsorted
  "properties": { "write.parquet.compression-codec": "zstd" },
  "current-snapshot-id": -1,  ← NO DATA YET
  "snapshots": [],             ← empty — first append() adds an entry here
  "metadata-log": []           ← history of previous metadata files
}
```
> After `table.append(batch_1)` — `current-snapshot-id` becomes a real 64-bit integer and `snapshots` gets one entry pointing to the manifest list Avro file on S3.

**Expected output:**
```
format_version : 2
table_uuid     : 6cf047cd-...
location       : s3://warehouse/pyiceberg_demo/sensor_readings
current_snap   : None   ← no data written yet
```
> **Try it:** Open MinIO at [localhost:9001](http://localhost:9001) → `warehouse/pyiceberg_demo/sensor_readings/metadata/` — you should see exactly one file: `00000-<uuid>.metadata.json`

---
## 4. Write Path — Append Data

The write path is the heart of this workshop. When you call `table.append(df)`, PyIceberg goes through 6 stages:

```
Stage 1: Transaction open              (IN MEMORY — nothing written)
Stage 2: Schema validation             (IN MEMORY — PyArrow ↔ Iceberg check)
Stage 3: DataFrame → Parquet → S3      (S3 WRITE #1 — your actual data)
Stage 4: Manifest file → S3            (S3 WRITE #2 — file index with stats)
Stage 5: Manifest list + Snapshot → S3 (S3 WRITE #3 — snapshot TOC)
Stage 6: Catalog commit                (ATOMIC GATE — HTTP POST)
```

**Nothing is visible to readers until Stage 6 completes.**

### Generate sample data

**What we're doing:** Creating a helper function that generates synthetic Picarro sensor data. Each batch = 50 rows with random sensor IDs, timestamps, CO2, and CH4 readings.

In [ ]:
import pyarrow as pa
import datetime
import random

def make_batch(batch_id: int, n_rows: int = 50) -> pa.Table:
    """Generate a batch of synthetic sensor readings."""
    base_ts = datetime.datetime(2024, 1, batch_id, 0, 0, 0)
    return pa.table({
        "sensor_id": [f"PICARRO-{random.randint(1,5):03d}" for _ in range(n_rows)],
        "ts":        [base_ts + datetime.timedelta(minutes=i) for i in range(n_rows)],
        "co2_ppm":   [400.0 + random.gauss(0, 5) for _ in range(n_rows)],
        "ch4_ppb":   [1850.0 + random.gauss(0, 20) for _ in range(n_rows)],
    })

# Preview one batch
sample = make_batch(1)
print(f"Batch shape: {sample.shape}")
print(f"Schema: {sample.schema}")
sample.to_pandas().head()

**Expected output:**
```
Batch shape: (50, 4)
Schema:
sensor_id: string
ts: timestamp[us]
co2_ppm: double
ch4_ppb: double
```
Plus a DataFrame preview with 5 rows of sensor data.

> **Note:** `pa.table()` creates a PyArrow Table — PyArrow infers `timestamp[us]` (microsecond precision) and `double` (64-bit float). During `table.append()`, Stage 2 validates these types match the Iceberg schema before any S3 write.

### Append 1 batch — watch the full write path happen once

**What we're doing:** Appending a single batch of 50 rows. One `table.append()` call runs all 6 stages of the write path: schema validation → Parquet write → manifest → manifest list → atomic catalog commit. After this cell: **1 snapshot, 1 Parquet file, 1 manifest, 1 manifest list.**

In [ ]:
total_rows = 0

batch = make_batch(1)
table.append(batch)
total_rows += len(batch)

snap = table.current_snapshot()
print(f"--- Batch 1 appended ({len(batch)} rows) ---")
print(f"  snapshot_id : {snap.snapshot_id}")
print(f"  timestamp   : {snap.timestamp_ms}")
print(f"  operation   : {snap.summary.operation.value}")
print(f"  added-data-files  : {snap.summary.get('added-data-files', 'N/A')}")
print(f"  added-records     : {snap.summary.get('added-records', 'N/A')}")
print(f"  total-records     : {snap.summary.get('total-records', 'N/A')}")
print(f"\nTotal rows written so far: {total_rows}")

**Expected output:**
```
--- Batch 1 appended (50 rows) ---
  snapshot_id       : 2985870393337822537   ← your ID will differ
  timestamp         : 1744099523000
  operation         : append
  added-data-files  : 1
  added-records     : 50
  total-records     : 50

Total rows written so far: 50
```

> **Check MinIO now:** `warehouse/pyiceberg_demo/sensor_readings/`
> - `data/` → **1** `.parquet` file
> - `metadata/` → **1** manifest (`.avro`) + **1** manifest list (`snap-*.avro`) + **2** `metadata.json` files

**Sequence diagram — what just happened inside `table.append(batch)`:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant TX as Transaction<br/><br/>table/__init__.py L209
    participant FA as _FastAppendFiles<br/><br/>table/update/snapshot.py L499
    participant PY as io/pyarrow.py<br/><br/>write_parquet()
    participant S3 as MinIO / S3
    participant MF as manifest.py
    participant CAT as REST Catalog Server

    Code->>TX: table.append(batch)
    TX->>FA: _FastAppendFiles initialized

    Note over TX,PY: Stage 1–2: Validate schema (client-side, no network)
    TX->>PY: _check_pyarrow_schema_compatible()
    PY-->>TX: ✅ Arrow dtypes match Iceberg types

    Note over PY,S3: Stage 3: Write Parquet — S3 write #1
    TX->>PY: bin_pack_arrow_table() → write_parquet()
    PY->>S3: PUT data/00000-0-uuid.parquet
    S3-->>PY: ✅ written
    PY-->>TX: DataFile + column stats (min/max, null counts)

    Note over MF,S3: Stage 4: Write Manifest file (Avro) — S3 write #2
    TX->>MF: ManifestEntry(DataFile)
    MF->>S3: PUT metadata/uuid-m0.avro
    S3-->>MF: ✅ written

    Note over FA,S3: Stage 5: Write Manifest List (Avro) — S3 write #3
    FA->>S3: PUT metadata/snap-id-uuid.avro
    S3-->>FA: ✅ written + Snapshot object created

    Note over TX,CAT: Stage 6: Atomic Catalog Commit — THE GATE
    TX->>CAT: POST /v1/tables/commit<br/>AddSnapshotUpdate + AssertRefSnapshotId
    CAT-->>TX: 200 OK — snapshot is now visible to all readers ✅
    TX-->>Code: return

    Note over TX,CAT: If another writer committed first → 409 Conflict (optimistic lock)
```

**What just happened (each append = 4 files + 1 atomic commit):**
- **Stage 1–2 (client-side):** `Transaction` opens, `_FastAppendFiles` initializes, Arrow dtypes validated against Iceberg schema — *no network yet*
- **Stage 3 — S3 write #1:** `write_parquet()` uploads the Parquet file and computes per-column `min/max/null` stats stored in the `DataFile` object
- **Stage 4 — S3 write #2:** `ManifestEntry` wraps the `DataFile` → written as Avro (`.avro`). This is how Iceberg indexes file-level stats for fast pruning
- **Stage 5 — S3 write #3:** `write_manifest_list()` writes the manifest list (another Avro file) pointing to all manifests, and creates a `Snapshot`
- **Stage 6 — Atomic commit:** `catalog.commit_table()` sends an HTTP POST with `AddSnapshotUpdate + AssertRefSnapshotId`. The `AssertRefSnapshotId` is the **optimistic lock** — if two writers race, only one wins; the other gets a `409 Conflict` and retries
- **Zero partial reads:** readers either see the old snapshot or the new one — never half-written data

### Now append 2 more batches — build up the snapshot chain

**What we're doing:** Appending batches 2 and 3 so we have 3 snapshots total. The Inspect section (Section 5) will walk this 3-snapshot chain. Each append is identical to what you just saw — same 6 stages, new snapshot, old ones untouched.

In [ ]:
for b in range(2, 4):
    batch = make_batch(b)
    table.append(batch)
    total_rows += len(batch)
    snap = table.current_snapshot()
    print(f"--- Batch {b} appended ({len(batch)} rows) | total-records: {snap.summary.get('total-records', 'N/A')} ---")

print(f"\nTotal rows written: {total_rows}")
print(f"Snapshots: {len(table.history())}")

**Expected output:**
```
--- Batch 2 appended (50 rows) | total-records: 100 ---
--- Batch 3 appended (50 rows) | total-records: 150 ---

Total rows written: 150
Snapshots: 3
```

> **Check MinIO:** `data/` → 3 `.parquet` files | `metadata/` → 3 manifests + 3 manifest lists + 4 `metadata.json` files

---
## 5. Inspect What Got Written

Now let's peel back the layers and look at the metadata tree Iceberg built during those 3 appends.

### 5a. Snapshot History

**What we're doing:** Listing all snapshots — each append created one. Snapshots are immutable; once created, never modified.

In [ ]:
print(f"Snapshot History ({len(table.history())} snapshots)")
print("=" * 60)
for h in table.history():
    ts = datetime.datetime.fromtimestamp(h.timestamp_ms / 1000)
    print(f"  {h.snapshot_id}  @  {ts}")

**Expected output:**
```
Snapshot History (3 snapshots)
============================================================
  2985870393337822537  @  2026-04-08 10:15:23
  5738291047382910473  @  2026-04-08 10:15:24
  8291047538291047538  @  2026-04-08 10:15:25
```

### 5b. Manifest List & Manifest Files

**What we're doing:** Walking the metadata tree: Snapshot → Manifest List → Manifest Files → Data Files. This is exactly what the read path does (but with pruning).

In [ ]:
snap = table.current_snapshot()
io = table.io

print(f"Current snapshot: {snap.snapshot_id}")
print(f"Manifest list:   {snap.manifest_list.split('/')[-1]}")
print()

manifest_list = snap.manifests(io)
print(f"Manifest files: {len(manifest_list)}")
print("=" * 60)

for i, mf in enumerate(manifest_list, 1):
    entries = mf.fetch_manifest_entry(io, discard_deleted=True)
    print(f"\n  [{i}] {mf.manifest_path.split('/')[-1]}")
    print(f"      added_files   : {mf.added_files_count}")
    print(f"      existing_files: {mf.existing_files_count}")
    print(f"      deleted_files : {mf.deleted_files_count}")
    for e in entries:
        df = e.data_file
        print(f"      data file: {df.file_path.split('/')[-1]}  ({df.record_count} rows, {df.file_size_in_bytes} bytes)")

**Expected output:**
```
Manifest files: 3
  [1] <uuid>-m0.avro  (batch 3) → 1 data file, 50 rows
  [2] <uuid>-m0.avro  (batch 2) → 1 data file, 50 rows
  [3] <uuid>-m0.avro  (batch 1) → 1 data file, 50 rows
```

### 5c. Data File Statistics

**What we're doing:** Inspecting column-level min/max bounds and null counts stored in a manifest entry. These are what enable file-level pruning during reads.

In [ ]:
# Inspect stats from the latest manifest's first data file
latest_manifest = manifest_list[0]
entries = latest_manifest.fetch_manifest_entry(io, discard_deleted=True)
data_file = entries[0].data_file

print(f"File: {data_file.file_path.split('/')[-1]}")
print(f"Rows: {data_file.record_count}")
print(f"Size: {data_file.file_size_in_bytes} bytes")
print(f"\nColumn stats (used for file-level pruning):")
print(f"  lower_bounds: {data_file.lower_bounds}")
print(f"  upper_bounds: {data_file.upper_bounds}")
print(f"  null_counts:  {data_file.null_value_counts}")
print(f"  value_counts: {data_file.value_counts}")

**Expected output:**
```
File: 00000-0-<uuid>.parquet
Rows: 50
Size: ~3500 bytes

Column stats (keys = field IDs):
  lower_bounds: {1: b'PICARRO-001', 2: b'...', 3: b'...', 4: b'...'}
  upper_bounds: {1: b'PICARRO-005', 2: b'...', 3: b'...', 4: b'...'}
  null_counts:  {1: 0, 2: 0, 3: 0, 4: 0}
  value_counts: {1: 50, 2: 50, 3: 50, 4: 50}
```

---
## 6. Read Path — Scan & Filter

The read path is a **pruning funnel** — each layer eliminates more work:

```
table.scan()  →  Snapshot resolution       (which snapshot to read?)
              →  Manifest-level pruning     (skip manifests by partition summary)
              →  File-level pruning         (skip files by column min/max)
              →  Row-group predicate pushdown (Parquet internal filtering)
              →  Column projection          (only read requested columns)
              →  Arrow Table returned
```

### 6a. Full table scan

**What we're doing:** Reading all data with no filters — every data file will be read in full.

In [ ]:
df = table.scan().to_arrow()
print(f"Total rows: {len(df)}  (expected {total_rows})")
print(f"Columns:    {df.column_names}")
df.to_pandas().head()

**Expected output:**
```
Total rows: 150  (expected 150)
Columns:    ['sensor_id', 'ts', 'co2_ppm', 'ch4_ppb']
```

**Sequence diagram — what happens under the hood:**

```mermaid
sequenceDiagram
    participant Code as Your Code
    participant DS as DataScan<br/><br/>table/__init__.py L1917
    participant ME as _ManifestEvalVisitor<br/><br/>expressions/visitors.py L546
    participant IE as _InclusiveMetricsEvaluator<br/><br/>expressions/visitors.py L1157
    participant S3 as MinIO / S3

    Code->>DS: table.scan().to_arrow()
    DS->>S3: fetch current snapshot's manifest list (.avro)
    S3-->>DS: ManifestList → [manifest1, manifest2, manifest3]

    Note over DS,IE: Pruning funnel (no filter → all pass)

    DS->>ME: _ManifestEvalVisitor: check each manifest
    ME-->>DS: ✅ all 3 manifests pass (no partition filter)

    DS->>IE: _InclusiveMetricsEvaluator: check file stats
    IE-->>DS: ✅ all 3 files pass (no row filter)

    DS->>S3: read file1.parquet + file2.parquet + file3.parquet (parallel)
    S3-->>DS: 3 × Arrow RecordBatch

    DS-->>Code: Combined Arrow Table (150 rows)
```

**What just happened:**
- `Table.scan()` creates a `DataScan` object — nothing is read yet (lazy)
- `.to_arrow()` triggers `plan_files()` → `scan_plan_helper()` which resolves the current snapshot and loads the manifest list from S3
- **Manifest pruning** (`_ManifestEvalVisitor`): checks partition-level stats — no filter here so all 3 manifests pass
- **File stats pruning** (`_InclusiveMetricsEvaluator`): checks column-level `min/max` bounds — no filter so all 3 files pass
- All 3 Parquet files are read **in parallel** via `ExecutorFactory`, then combined into one Arrow Table
- On a real table with 10,000 files and a tight filter, steps 2 and 3 might prune to just 2 files — **without opening a single Parquet file**

### 6b. Filtered scan (predicate pushdown)

**What we're doing:** Adding a row filter `co2_ppm > 405`. Iceberg will use column statistics to potentially skip entire files.

In [ ]:
# Filter: only rows where co2_ppm > 405
filtered = table.scan(row_filter="co2_ppm > 405").to_arrow()
print(f"Rows with co2_ppm > 405: {len(filtered)} out of {total_rows}")
filtered.to_pandas().head()

**Expected output:**
```
Rows with co2_ppm > 405: ~35-45 out of 150    ← varies (random data)
```

**What just happened:**
- The string `"co2_ppm > 405"` is parsed into `GreaterThan(Reference("co2_ppm"), Literal(405.0))`, then `bind()` resolves `"co2_ppm"` → **field ID 3**
- `_InclusiveMetricsEvaluator` checks each file's `upper_bounds[3]` (the max `co2_ppm` in that file) — if the max ≤ 405, the **entire file is skipped without opening it**
- Files that pass get a second round of filtering inside Parquet via PyArrow's row-group predicate pushdown
- Two layers of pruning: **file-level** (free — stats in Avro) + **row-group-level** (inside Parquet)

### 6c. Column projection (read only specific fields)

**What we're doing:** Requesting only 2 of 4 columns. Iceberg tells Parquet to skip reading `ts` and `ch4_ppb` entirely.

In [ ]:
# Only read sensor_id and co2_ppm — Parquet column pruning kicks in
projected = table.scan(
    selected_fields=("sensor_id", "co2_ppm")
).to_arrow()
print(f"Columns returned: {projected.column_names}")
print(f"Rows: {len(projected)}")
projected.to_pandas().head()

**Expected output:**
```
Columns returned: ['sensor_id', 'co2_ppm']
Rows: 150
```

**What just happened:**
- `DataScan.projection()` builds a projected `Schema` containing only field IDs `1` (sensor_id) and `3` (co2_ppm)
- This schema is passed to PyArrow's Parquet reader as a column selection
- Parquet is **columnar** — only the requested column chunks are loaded from S3; `ts` and `ch4_ppb` are never touched
- On a wide table (say 200 columns), selecting 3 columns saves ~98.5% of S3 I/O

### 6d. Time travel — query an older snapshot

**What we're doing:** Reading the table as it was after batch 1 only (50 rows), by passing the first snapshot's ID.

In [ ]:
# Get the first snapshot (after batch 1 only)
first_snapshot_id = table.history()[0].snapshot_id
print(f"First snapshot ID: {first_snapshot_id}")

# Time travel: read table as it was after batch 1
old_df = table.scan(snapshot_id=first_snapshot_id).to_arrow()
print(f"Rows at first snapshot: {len(old_df)}  (expected 50)")

# Compare with current
current_df = table.scan().to_arrow()
print(f"Rows at current snapshot: {len(current_df)}  (expected {total_rows})")

**Expected output:**
```
First snapshot ID: 2985870393337822537
Rows at first snapshot:   50   (expected 50)
Rows at current snapshot: 150  (expected 150)
```

**What just happened:**
- `DataScan(snapshot_id=first_snapshot_id)` looks up that snapshot in `metadata.snapshots[]`
- That older snapshot has its own manifest list pointing to only 1 manifest → 1 data file (50 rows)
- The same read pipeline runs, but against that snapshot's manifest chain
- **No data was copied or moved** — the old Parquet file is still sitting on S3, referenced by the old snapshot
- `expire_snapshots()` removes old metadata entries; `delete_orphan_files()` removes unreferenced Parquet files

---
## 7. Schema Evolution

**What we're doing:** Adding a new column `humidity` to the table — **without rewriting any data files**.

In [ ]:
from pyiceberg.types import FloatType

with table.update_schema() as update:
    update.add_column("humidity", FloatType(), doc="Relative humidity %")

print("Schema after evolution:")
for field in table.schema().fields:
    print(f"  {field.field_id}: {field.name} ({field.field_type})")

**Expected output:**
```
Schema after evolution:
  1: sensor_id (string)
  2: ts (timestamp)
  3: co2_ppm (double)
  4: ch4_ppb (double)
  5: humidity (float)
```

**What just happened:**
- `update_schema()` returns an `UpdateSchema` context manager — `add_column("humidity", FloatType())` assigns it the **next available field ID: 5**
- On `__exit__`, a `Transaction` commits the schema change: new `metadata.json` is written to S3 and registered via `catalog.commit_table()`
- **Zero data files were touched** — existing Parquet files (fields 1–4) are unchanged
- When you read old rows with the evolved schema, Iceberg projects field 5 → `null` for all old rows
- This is what makes Iceberg schema evolution **safe and backward-compatible**

### Read old data with the evolved schema

**What we're doing:** Reading the 150 existing rows — they were written before `humidity` existed. Iceberg fills `null` for the missing column.

In [ ]:
df_evolved = table.scan().to_arrow()
print(f"Columns: {df_evolved.column_names}")
print(f"Humidity nulls: {df_evolved['humidity'].null_count} out of {len(df_evolved)}")
df_evolved.to_pandas().head()

**Expected output:**
```
Columns: ['sensor_id', 'ts', 'co2_ppm', 'ch4_ppb', 'humidity']
Humidity nulls: 150 out of 150   ← existing rows fill null; zero rewrites
```

**What just happened:**
- `project_table()` in `io/pyarrow.py` processes each Parquet file: reads the columns that exist (fields 1–4), then **creates a null array** of the right length for field 5 (`humidity`)
- Old and new data are assembled into the same projected Arrow Table with 5 columns
- New batches appended *after* the schema change will have real `humidity` values; old rows stay `null`
- Iceberg resolves columns by **field ID, not name or position** — this is why renaming a column never breaks old files

---
## 8. Snapshot Management

**What we're doing:** Using the `table.inspect` API to query metadata as Arrow tables — useful for auditing and monitoring.

### Inspect snapshots

In [ ]:
# Snapshots table — one row per snapshot
snapshots_df = table.inspect.snapshots().to_pandas()
print("Snapshots:")
snapshots_df

**Expected output:** A table with ~4 rows (3 appends + 1 schema update) — each row shows `snapshot_id`, `committed_at`, `operation`, and `parent_id` (the chain).

**What just happened:**
- `InspectTable.snapshots()` reads `metadata.snapshots[]` from the current `metadata.json` and converts it to an Arrow Table
- You can see the full snapshot chain: each snapshot's `parent_id` points to the previous one — this is the immutable history log
- Operations: `append` (data added), `overwrite` (replace), `delete` (row-level), `replace` (compaction/rewrite)

### Inspect manifests

In [ ]:
# Manifests table — one row per manifest file
manifests_df = table.inspect.manifests().to_pandas()
print("Manifests:")
manifests_df

**Expected output:** A table with 3 rows — one manifest file per append — showing `path`, `length`, `added_data_files_count`, `existing_data_files_count`.

**What just happened:**
- `InspectTable.manifests()` loads the current snapshot's manifest list, then returns one row per manifest file
- `added_data_files_count`: files this manifest contributed to the snapshot
- `existing_data_files_count`: files inherited from previous snapshots (Iceberg never rewrites old manifests)
- In production this table helps you decide when to run `rewrite_manifests()` (compaction of many small manifests)

### Inspect data files

In [ ]:
# Data files table — one row per Parquet file
files_df = table.inspect.files().to_pandas()
print("Data files:")
files_df[["file_path", "record_count", "file_size_in_bytes", "file_format"]]

**Expected output:**
```
Data files:
  file_path                                record_count  file_size  format
  s3://.../00000-0-<uuid>.parquet               50        ~3500     PARQUET
  s3://.../00000-0-<uuid>.parquet               50        ~3500     PARQUET
  s3://.../00000-0-<uuid>.parquet               50        ~3500     PARQUET
```

**What just happened:**
- `InspectTable.files()` opens all manifest files for the current snapshot and collects every `DataFile` entry — one row per Parquet file
- Columns include `file_path`, `record_count`, `file_size_in_bytes`, `column_sizes`, `value_counts`, `null_value_counts`, `lower_bounds`, `upper_bounds`
- These are exactly the stats `_InclusiveMetricsEvaluator` uses during read-path pruning
- In production, if you see many small files (e.g. 10,000 files × 50 rows), run `table.maintenance().rewrite_data_files()` to compact them

---
## Summary

| What we did | What happened under the hood |
|---|---|
| `catalog.create_table()` | `POST /v1/.../tables` → metadata.json to S3, no data files |
| `table.append(batch)` | 6 stages: Parquet→S3, Manifest→S3, ManifestList→S3, `commit_table()` HTTP POST |
| `table.scan(row_filter=...)` | `_ManifestEvalVisitor` → `_InclusiveMetricsEvaluator` → Parquet pushdown |
| `table.scan(snapshot_id=...)` | Resolves different snapshot → different manifest list → different data files |
| `table.update_schema()` | New metadata.json only — `project_table()` fills nulls for old files |
| `table.inspect.*` | Reads metadata structures and returns them as Arrow tables |

### Key Takeaways

1. **Write path = 6 stages, atomic at the end.** Code: `Table.append()` → `Transaction` → `_FastAppendFiles` → `commit_table()`
2. **Read path = a pruning funnel.** Code: `DataScan.plan_files()` → `_ManifestEvalVisitor` → `_InclusiveMetricsEvaluator`
3. **Schema evolution is free.** Code: `UpdateSchema` → new metadata.json → `project_table()` handles null backfill
4. **Time travel is built in.** Snapshots are immutable; scan resolves a different entry point in the metadata tree
5. **ACID via optimistic concurrency.** Code: `AssertRefSnapshotId` in `commit_table()` → 409 on conflict → caller must `refresh()` + retry

---
*Workshop by Vipin Kataria, Picarro — April 2026*